In [2]:
import requests
import pandas as pd
import time

coins = ["bitcoin", "ethereum", "litecoin", "ripple"]
all_data = []

for coin in coins:
    print(f"Fetching {coin}...")

    url = f"https://api.coingecko.com/api/v3/coins/{coin}/market_chart"
    params = {
        "vs_currency": "usd",
        "days": "90"
    }

    response = requests.get(url, params=params)

    # Check status
    if response.status_code != 200:
        print(f"Error for {coin}: {response.status_code}")
        continue

    data = response.json()

    # Check if 'prices' exists
    if "prices" not in data:
        print(f"Missing 'prices' for {coin}: {data}")
        continue

    df = pd.DataFrame(data["prices"], columns=["timestamp", "price"])
    df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    df["coin"] = coin

    all_data.append(df)

    time.sleep(2)  # IMPORTANT: increase delay

# Combine
final_df = pd.concat(all_data, ignore_index=True)

final_df.head()

Fetching bitcoin...
Fetching ethereum...
Fetching litecoin...
Fetching ripple...


,timestamp,price,date,coin
0,1766296928076,88096.923325,2025-12-21 06:02:08.076,bitcoin
1,1766300488981,88067.806506,2025-12-21 07:01:28.981,bitcoin
2,1766304136261,88096.039525,2025-12-21 08:02:16.261,bitcoin
3,1766307672234,88507.341770,2025-12-21 09:01:12.234,bitcoin
4,1766311212262,88948.664173,2025-12-21 10:00:12.262,bitcoin


In [3]:
final_df.shape

(8647, 4)

In [4]:
# Remove timestamp
final_df.drop(columns=["timestamp"], inplace=True)

# Ensure correct types
final_df["price"] = final_df["price"].astype(float)

# Sort data
final_df.sort_values(by="date", inplace=True)

In [5]:
df = pd.DataFrame(data["prices"], columns=["timestamp", "price"])

volumes = pd.DataFrame(data["total_volumes"], columns=["timestamp", "volume"])

df["volume"] = volumes["volume"]

In [6]:
# Check missing values
final_df.isnull().sum()

,0
price,0
date,0
coin,0


In [7]:
final_df.to_csv("crypto_data_API.csv", index=False)